In [37]:
import numpy as np
import pandas as pd

#  Charger les données
df = pd.read_csv("data.csv")

if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])

if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Transformer la cible
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

X = df.drop(columns=['diagnosis']).values.astype(float)
y = df['diagnosis'].values.astype(int)


#  Train / Test split
np.random.seed(42)

indices = np.arange(len(X))
np.random.shuffle(indices)

split = int(0.8 * len(X))

X_train = X[indices[:split]]
y_train = y[indices[:split]]

X_test = X[indices[split:]]
y_test = y[indices[split:]]


In [38]:
#  Arbre de décision 
# Un Decision Stump est un arbre de décision très simple.
# Il utilise une seule variable et un seul seuil pour séparer les données.
#
# Exemple :
# si radius_mean <= 15.2  -> classe 0
# sinon                  -> classe 1

class DecisionStump:

    def fit(self, X, y):
        # X contient les variables explicatives
        # y contient les classes réelles : 0 ou 1

        # n_samples = nombre de lignes
        # n_features = nombre de variables
        n_samples, n_features = X.shape

        # On initialise la meilleure erreur avec une valeur très grande
        # Le but est de trouver le seuil qui donne l'erreur la plus faible
        best_error = float('inf')

        # Dans cette version simple de Random Forest,
        # chaque arbre choisit une variable aléatoire
        feature = np.random.randint(0, n_features)

        # On récupère toutes les valeurs possibles de cette variable
        # Chaque valeur peut être testée comme seuil de séparation
        thresholds = np.unique(X[:, feature])

        # Tester chaque seuil possible
        for t in thresholds:

            # Groupe gauche :
            # les observations où la variable est <= seuil
            left = X[:, feature] <= t

            # Groupe droit :
            # les observations où la variable est > seuil
            right = X[:, feature] > t

            # Si un des groupes est vide, la séparation est inutile
            if np.sum(left) == 0 or np.sum(right) == 0:
                continue

            # Classe majoritaire dans le groupe gauche
            # Si la majorité des valeurs y est 1, alors classe gauche = 1
            # Sinon classe gauche = 0
            left_class = 1 if np.sum(y[left]) > len(y[left]) / 2 else 0

            # Classe majoritaire dans le groupe droit
            right_class = 1 if np.sum(y[right]) > len(y[right]) / 2 else 0

            # Créer un tableau pour stocker les prédictions
            y_pred = np.zeros(n_samples)

            # Les points du groupe gauche prennent la classe gauche
            y_pred[left] = left_class

            # Les points du groupe droit prennent la classe droite
            y_pred[right] = right_class

            # Calculer le nombre d'erreurs
            # On compare les prédictions avec les vraies classes
            error = np.sum(y != y_pred)

            # Si cette séparation est meilleure que les précédentes,
            # on sauvegarde cette variable, ce seuil et les deux classes
            if error < best_error:
                best_error = error
                self.feature = feature
                self.threshold = t
                self.left_class = left_class
                self.right_class = right_class

    def predict(self, X):
        # Cette fonction prédit les classes de nouvelles données

        # Créer un tableau vide pour les prédictions
        y_pred = np.zeros(X.shape[0])

        # Appliquer la règle apprise pendant l'entraînement
        left = X[:, self.feature] <= self.threshold
        right = X[:, self.feature] > self.threshold

        # Si l'observation est à gauche du seuil,
        # elle prend la classe gauche
        y_pred[left] = self.left_class

        # Sinon, elle prend la classe droite
        y_pred[right] = self.right_class

        return y_pred

In [39]:
class SimpleRandomForest:
    
    def __init__(self, n_trees=10):
        # n_trees = nombre d’arbres dans la forêt
        self.n_trees = n_trees
        
        # liste qui va contenir tous les arbres (stumps)
        self.trees = []

    def bootstrap(self, X, y):
        # Bootstrap sampling
        # On crée un échantillon aléatoire AVEC remplacement
        # → certains points peuvent être répétés
        # → certains points peuvent ne pas être choisis

        n_samples = X.shape[0]

        # Tirage aléatoire d’indices
        idx = np.random.choice(n_samples, n_samples, replace=True)

        # On retourne les nouvelles données échantillonnées
        return X[idx], y[idx]

    def fit(self, X, y):
        #  Entraînement de la forêt

        # Réinitialiser la liste des arbres
        self.trees = []

        # Créer plusieurs arbres
        for _ in range(self.n_trees):

            # Créer un arbre simple (stump)
            stump = DecisionStump()

            # Générer un dataset aléatoire (bootstrap)
            X_sample, y_sample = self.bootstrap(X, y)

            # Entraîner l’arbre sur cet échantillon
            stump.fit(X_sample, y_sample)

            # Ajouter l’arbre à la forêt
            self.trees.append(stump)

    def predict(self, X):
        # Prédiction

        # Liste pour stocker les prédictions de chaque arbre
        predictions = []

        # Chaque arbre fait une prédiction
        for tree in self.trees:
            predictions.append(tree.predict(X))

        # Transformer en tableau numpy
        # shape = (nombre d’arbres, nombre de données)
        predictions = np.array(predictions)

        # Vote majoritaire

        final_pred = []

        # Pour chaque observation (colonne)
        for i in range(X.shape[0]):

            # On récupère toutes les prédictions des arbres
            # predictions[:, i] = prédictions de tous les arbres pour la donnée i
            values, counts = np.unique(predictions[:, i], return_counts=True)

            # Choisir la classe la plus fréquente (vote)
            final_pred.append(values[np.argmax(counts)])

        # Retourner les prédictions finales
        return np.array(final_pred)

In [40]:
#  Entraîner
model = SimpleRandomForest(n_trees=10)
model.fit(X_train, y_train)


# Prédictions
y_pred = model.predict(X_test)


#  Évaluation simple
correct = 0
incorrect = 0

for i in range(len(y_test)):
    if y_test[i] == y_pred[i]:
        correct += 1
    else:
        incorrect += 1

print("Résultat Random Forest SIMPLE :")
print("Correct :", correct)
print("Incorrect :", incorrect)

Résultat Random Forest SIMPLE :
Correct : 84
Incorrect : 30
